# Credit Card Fraud Detection - Exploratory Data Analysis

Explore credit card transaction data and analyze fraud patterns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

df = pd.read_csv('../data/creditcard.csv')
print(f"Dataset shape: {df.shape}")
print(f"Memory usage: {df.memory_usage().sum() / 1024**2:.2f} MB")
df.head()

## Dataset Overview

In [ ]:
print("Dataset Info:")
print(df.info())
print("\nBasic Statistics:")
print(df.describe())

# Get class/fraud column
fraud_col = [col for col in df.columns if 'class' in col.lower() or 'fraud' in col.lower()][0] if any('class' in col.lower() or 'fraud' in col.lower() for col in df.columns) else df.columns[-1]

## Class Imbalance Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
fraud_counts = df[fraud_col].value_counts()
axes[0].bar(['Legitimate', 'Fraud'], fraud_counts.sort_index().values, color=['#2ca02c', '#d62728'])
axes[0].set_ylabel('Count')
axes[0].set_title('Transaction Class Distribution')
axes[0].set_yscale('log')

# Pie chart
axes[1].pie(fraud_counts.sort_index().values, labels=['Legitimate', 'Fraud'], autopct='%1.3f%%',
            colors=['#2ca02c', '#d62728'])
axes[1].set_title('Fraud Rate')

plt.tight_layout()
plt.show()

fraud_rate = (df[fraud_col] == 1).sum() / len(df) * 100
print(f"Fraud Rate: {fraud_rate:.3f}%")
print(f"Fraud Cases: {(df[fraud_col] == 1).sum()}")
print(f"Legitimate Cases: {(df[fraud_col] == 0).sum()}")

## Amount Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Amount distribution
axes[0, 0].hist([df[df[fraud_col]==0]['Amount'],
                 df[df[fraud_col]==1]['Amount']],
               label=['Legitimate', 'Fraud'],
               bins=50, color=['#2ca02c', '#d62728'])
axes[0, 0].set_xlabel('Amount ($)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Transaction Amount Distribution')
axes[0, 0].legend()
axes[0, 0].set_yscale('log')

# Amount boxplot
data_to_plot = [df[df[fraud_col]==0]['Amount'], df[df[fraud_col]==1]['Amount']]
axes[0, 1].boxplot(data_to_plot, labels=['Legitimate', 'Fraud'])
axes[0, 1].set_ylabel('Amount ($)')
axes[0, 1].set_title('Amount Comparison by Class')

# Log amount distribution
df['Log_Amount'] = np.log10(df['Amount'] + 1)
axes[1, 0].hist([df[df[fraud_col]==0]['Log_Amount'],
                 df[df[fraud_col]==1]['Log_Amount']],
               label=['Legitimate', 'Fraud'],
               bins=50, color=['#2ca02c', '#d62728'])
axes[1, 0].set_xlabel('Log Amount')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Log-Scaled Amount Distribution')
axes[1, 0].legend()

axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

print(f"Legitimate Mean Amount: ${df[df[fraud_col]==0]['Amount'].mean():.2f}")
print(f"Fraud Mean Amount: ${df[df[fraud_col]==1]['Amount'].mean():.2f}")

## Feature Analysis

In [ ]:
# Get V columns (anonymized PCA features)
v_cols = [col for col in df.columns if col.startswith('V')]

# Correlation with fraud
correlations = []
for col in v_cols:
    corr = df[col].corr(df[fraud_col])
    correlations.append({'Feature': col, 'Correlation': abs(corr)})

corr_df = pd.DataFrame(correlations).sort_values('Correlation', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(20), corr_df.head(20)['Correlation'].values)
ax.set_yticks(range(20))
ax.set_yticklabels(corr_df.head(20)['Feature'].values)
ax.set_xlabel('Absolute Correlation with Fraud')
ax.set_title('Top 20 Features Correlated with Fraud')
plt.tight_layout()
plt.show()

## Key Insights

1. **Severe Class Imbalance**: Fraud comprises ~0.17% of transactions
2. **Amount Patterns**: Fraudulent transactions tend to have different amount distributions
3. **Feature Variations**: Different V-features show varying discriminative power
4. **Data Anonymization**: Features are PCA-transformed, making direct interpretation difficult
5. **Sampling Strategy**: SMOTE will be essential for model training